In [111]:
import torch.nn as nn
import math

In [5]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=1, bias=False, attn_dropout=0., proj_dropout=0.):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.d = math.sqrt(self.head_dim)
        
        #Input dim?
        self.qkv = nn.Linear(dim, dim * 3, bias=bias)
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.proj = nn.Linear(dim, dim)
        self.proj_dropout = nn.Dropout(proj_dropout)
    
    # X has shape [B, D]
    def forward(self, x):
        batch_size, _ = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(batch_size, self.num_heads, 3 * self.head_dim)
#         qkv = qkv.permute(0, 

In [231]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        assert dim % num_heads == 0, 'dim should be divisible by num_heads'
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        print(B, N, C)
        qkv = self.qkv(x)
        print(qkv)
        qkv = qkv.reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        print(qkv.shape)
        q, k, v = qkv.unbind(0)   # make torchscript happy (cannot use tensor as tuple)
        print(q.shape)
#         print(q)
        print(k.transpose(-2, -1).shape)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        print(attn.shape)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [232]:
num_heads = 2
embed_dim = 256
head_dim = embed_dim // num_heads
batch_size = 128
x = torch.rand(batch_size, input_dim)

In [233]:
a = Attention(embed_dim, num_heads)
timm_out = a(x.unsqueeze(0))
timm_out

torch.Size([1, 128, 256])
1 128 256
tensor([[[ 0.0834,  0.5561, -0.4815,  ..., -0.0334, -0.3758,  0.3566],
         [ 0.1436,  0.4734, -0.8345,  ..., -0.3381, -0.1002,  0.4295],
         [ 0.4628,  0.5370, -0.5932,  ..., -0.1753, -0.4079,  0.4036],
         ...,
         [ 0.4371,  0.4856, -0.4904,  ..., -0.4451, -0.2881,  0.0259],
         [ 0.4251,  0.5601, -0.8418,  ..., -0.3078,  0.1673,  0.2471],
         [-0.1363,  0.0666, -0.5889,  ..., -0.0339, -0.5414,  0.4163]]],
       grad_fn=<UnsafeViewBackward0>)
torch.Size([3, 1, 2, 128, 128])
torch.Size([1, 2, 128, 128])
torch.Size([1, 2, 128, 128])
torch.Size([1, 2, 128, 128])


tensor([[[-0.0571,  0.2837,  0.0292,  ..., -0.1648,  0.3982,  0.0243],
         [-0.0576,  0.2837,  0.0295,  ..., -0.1654,  0.3977,  0.0239],
         [-0.0577,  0.2846,  0.0296,  ..., -0.1652,  0.3974,  0.0234],
         ...,
         [-0.0569,  0.2842,  0.0293,  ..., -0.1649,  0.3978,  0.0241],
         [-0.0582,  0.2837,  0.0293,  ..., -0.1651,  0.3972,  0.0238],
         [-0.0575,  0.2837,  0.0292,  ..., -0.1658,  0.3975,  0.0243]]],
       grad_fn=<ViewBackward0>)

In [227]:
qkv = a.qkv
proj = a.proj

In [234]:
# print("Input dim:", x.shape)
out = qkv(x)
# print("QKV output dim:", out.shape)
out = out.reshape(batch_size, num_heads, 3 * head_dim)
print(out)
print(out.shape)
# print("Reshaped output:", out.shape)
out = out.permute(1, 0, 2)
print("Permuted output:", out.shape)
q, k, v = out.chunk(3, dim=-1)
# print(q)
# print("Queries shape:", q.shape)
weights = torch.bmm(q, k.transpose(-2, -1)) / math.sqrt(head_dim)
# print(k.transpose(-2, -1).shape)
# print("Attn weights", weights.shape)
attn = weights.softmax(dim=-1)
# print("Softmax attention:", attn.shape)
out = torch.bmm(attn, v)
# print("Attention out:", out.shape)
out = out.transpose(0, 1).reshape(batch_size, num_heads * head_dim)
# print("Proj input", out.shape)
out = proj(out)
# print("Final out:", out.shape)
out

tensor([[[ 0.4733,  0.1895, -0.1895,  ..., -0.2431,  0.2359,  0.0517],
         [ 0.2762, -0.1239, -0.5698,  ...,  0.3371, -0.1038,  0.2176]],

        [[ 0.8196,  0.1907, -0.0974,  ..., -0.3005,  0.4227, -0.1061],
         [ 0.2366, -0.1728, -0.6576,  ...,  0.1025, -0.1006,  0.3269]],

        [[ 0.7234,  0.3267, -0.4158,  ..., -0.1202,  0.7616,  0.2557],
         [ 0.5308, -0.2497, -0.5589,  ...,  0.1152,  0.0411,  0.0607]],

        ...,

        [[ 0.9335,  0.2812, -0.1119,  ..., -0.4277,  0.5180, -0.0420],
         [ 0.4972, -0.0810, -0.4296,  ..., -0.0879,  0.0651,  0.2084]],

        [[ 0.4940,  0.4123, -0.2734,  ..., -0.2246,  0.3368, -0.0670],
         [ 0.2204, -0.1525, -0.3996,  ...,  0.0920, -0.0542,  0.3138]],

        [[ 0.7727,  0.2681, -0.6568,  ..., -0.3855,  0.2832,  0.0796],
         [ 0.6172, -0.4287, -0.3881,  ...,  0.3127, -0.0440,  0.3152]]],
       grad_fn=<ReshapeAliasBackward0>)
torch.Size([128, 2, 384])
Permuted output: torch.Size([2, 128, 384])


tensor([[ 0.0245, -0.0404,  0.1352,  ...,  0.0325, -0.2779,  0.1402],
        [ 0.0247, -0.0403,  0.1356,  ...,  0.0328, -0.2783,  0.1403],
        [ 0.0247, -0.0403,  0.1354,  ...,  0.0329, -0.2784,  0.1405],
        ...,
        [ 0.0244, -0.0401,  0.1353,  ...,  0.0331, -0.2782,  0.1406],
        [ 0.0245, -0.0407,  0.1357,  ...,  0.0331, -0.2782,  0.1403],
        [ 0.0242, -0.0403,  0.1349,  ...,  0.0329, -0.2785,  0.1403]],
       grad_fn=<AddmmBackward0>)

In [200]:
torch.allclose(out, timm_out)

True

In [164]:
queries = x
keys = x
values = x
true_output, _ = torch.nn.functional.multi_head_attention_forward(
            queries,
            keys,
            values,
            embed_dim_to_check=embed_dim,
            num_heads=num_heads,
            in_proj_weight=a.qkv.weight,
            in_proj_bias=a.qkv.bias,
            bias_k=None,
            bias_v=None,
            add_zero_attn=False,
            dropout_p=0.0,
            out_proj_bias=a.proj.bias,
            out_proj_weight=a.proj.weight,
            training=False,
            key_padding_mask=None,
            need_weights=True,
            attn_mask=None,
            use_separate_proj_weight=False,
            q_proj_weight=None,
            k_proj_weight=None,
            v_proj_weight=None,
            static_k=None,
            static_v=None,
        )
true_output

tensor([[-0.2650,  0.1402,  0.0698,  ...,  0.3949,  0.2181,  0.2912],
        [-0.2651,  0.1408,  0.0698,  ...,  0.3946,  0.2176,  0.2914],
        [-0.2653,  0.1406,  0.0696,  ...,  0.3958,  0.2175,  0.2915],
        ...,
        [-0.2649,  0.1411,  0.0696,  ...,  0.3945,  0.2173,  0.2919],
        [-0.2656,  0.1403,  0.0695,  ...,  0.3953,  0.2177,  0.2910],
        [-0.2647,  0.1407,  0.0695,  ...,  0.3951,  0.2171,  0.2903]],
       grad_fn=<SqueezeBackward1>)

In [165]:
torch.allclose(timm_out.squeeze()[0], true_output[0])

True

In [166]:
out

tensor([[0.0691, 0.0615, 0.2310,  ..., 0.0879, 0.0062, 0.0121],
        [0.0699, 0.0616, 0.2304,  ..., 0.0876, 0.0064, 0.0122],
        [0.0697, 0.0622, 0.2301,  ..., 0.0869, 0.0061, 0.0119],
        ...,
        [0.0698, 0.0618, 0.2307,  ..., 0.0868, 0.0062, 0.0125],
        [0.0695, 0.0613, 0.2306,  ..., 0.0880, 0.0058, 0.0120],
        [0.0694, 0.0618, 0.2303,  ..., 0.0872, 0.0062, 0.0121]],
       grad_fn=<AddmmBackward0>)